In [4]:
import warnings
import tracemalloc

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.preprocessing import (
    PowerTransformer, QuantileTransformer, MinMaxScaler, 
    StandardScaler, RobustScaler, KBinsDiscretizer
)
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.datasets import load_iris

from imblearn import FunctionSampler
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from feature_engine.outliers import Winsorizer, OutlierTrimmer

#warnings.filterwarnings('ignore')

In [17]:
def run_experiments(X_list, y_list, dataset_names):
    models = {
        'Gaussian Naive Bayes': GaussianNB(),
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=2000),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC(random_state=42, max_iter=2000),
        'Random Forest': RandomForestClassifier(random_state=42),
        'MLP': MLPClassifier(random_state=42, max_iter=2000)
    }

    def Trimmer(X, y, capping_method, fold, variables):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
            
        trimmer = OutlierTrimmer(capping_method=capping_method, tail='both', fold=fold, variables=variables)
        X_res = trimmer.fit_transform(X)
        
        y_was_array = isinstance(y, np.ndarray)
        if y_was_array:
            y = pd.Series(y, index=X.index)
            
        y_res = y.loc[X_res.index]
        
        if y_was_array:
            y_res = y_res.values
            
        return X_res, y_res

    metrics = [
        'balanced_accuracy', 
        'precision_macro', 
        'recall_macro', 
        'f1_macro', 
        'average_precision'
    ]
    
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    for X, y, name in zip(X_list, y_list, dataset_names):
        
        continuous_cols = [col for col in X.columns if not col.startswith('categorical_')]
        
        preprocessors = {
            'Baseline': 'passthrough',
            'IQR Capping': Winsorizer(capping_method='iqr', tail='both', fold=1.5, variables=continuous_cols),
            'Quantile Capping': Winsorizer(capping_method='quantiles', tail='both', fold=0.05, variables=continuous_cols),
            'IQR Removal': FunctionSampler(func=Trimmer, kw_args={'capping_method': 'iqr', 'fold': 1.5, 'variables': continuous_cols}, validate=False),
            'Quantile Removal': FunctionSampler(func=Trimmer, kw_args={'capping_method': 'iqr', 'fold': 1.5, 'variables': continuous_cols}, validate=False),
            'Yeo-Johnson': PowerTransformer(method='yeo-johnson'),
            'Quantile Transform': QuantileTransformer(output_distribution='normal', random_state=42),
            'Min-Max Scaler': MinMaxScaler(),
            'Standard Scaler': StandardScaler(),
            'Robust Scaler': RobustScaler(),
            'Uniform Binning': KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform', subsample=None),
            'Quantile Binning': KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile', subsample=None),
            'Random Undersampling': RandomUnderSampler(random_state=42),
            'SMOTE': SMOTE(random_state=42)
        }
        
        pbar = tqdm(total=len(models) * len(preprocessors), desc=f"Evaluating: {name}")

        for model_name, model in models.items():
            for prep_name, prep in preprocessors.items():
                
                steps = []
                
                if prep == 'passthrough':
                    steps.append(('preprocessor', 'passthrough'))
                elif isinstance(prep, (SMOTE, RandomUnderSampler, FunctionSampler)):
                    steps.append(('sampler', prep))
                else:
                    ct = ColumnTransformer(
                        transformers=[('num', prep, continuous_cols)],
                        remainder='passthrough'
                    )
                    steps.append(('preprocessor', ct))
                
                steps.append(('classifier', model))
                pipeline = Pipeline(steps)
                
                tracemalloc.start()
                
                try:
                    scores = cross_validate(
                        pipeline, X, y, 
                        cv=cv, scoring=metrics, n_jobs=-1, return_train_score=False
                    )
                    
                    _, peak = tracemalloc.get_traced_memory()
                    tracemalloc.stop()
                    
                    results.append({
                        'Dataset': name,
                        'Model': model_name,
                        'Preprocessor': prep_name,
                        'Fit Time (s)': np.mean(scores['fit_time']),
                        'Predict Time (s)': np.mean(scores['score_time']),
                        'Peak Memory (MB)': peak / (1024 * 1024),
                        'Balanced Accuracy': np.mean(scores['test_balanced_accuracy']),
                        'Precision Macro': np.mean(scores['test_precision_macro']),
                        'Recall Macro': np.mean(scores['test_recall_macro']),
                        'F1 Macro': np.mean(scores['test_f1_macro']),
                        'Average Precision': np.mean(scores['test_average_precision']),
                        'Error': None
                    })
                    
                except Exception as e:
                    tracemalloc.stop()
                    results.append({
                        'Dataset': name,
                        'Model': model_name,
                        'Preprocessor': prep_name,
                        'Fit Time (s)': np.nan,
                        'Predict Time (s)': np.nan,
                        'Peak Memory (MB)': np.nan,
                        'Balanced Accuracy': np.nan,
                        'Precision Macro': np.nan,
                        'Recall Macro': np.nan,
                        'F1 Macro': np.nan,
                        'Average Precision': np.nan,
                        'Error': str(e)
                    })
                
                pbar.update(1)
        
        pbar.close()
        
    return pd.DataFrame(results)

In [ ]:
df_bank = pd.read_csv('data/bank-additional-full.csv', sep=';')
categorical_cols_bank = ['job', 'marital', 'education', 'contact', 'month', 'day_of_week', 'poutcome', 'default', 'housing', 'loan']
df_bank = pd.get_dummies(df_bank, columns=categorical_cols_bank, drop_first=True, dtype=int, prefix='categorical', prefix_sep='_')
y_bank = df_bank['y'].map({'yes': 1, 'no': 0})
X_bank = df_bank.drop(columns=['y', 'duration'])

df_credit = pd.read_excel('data/default of credit card clients.xls', header=1)
categorical_cols_credit = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
df_credit = pd.get_dummies(df_credit, columns=categorical_cols_credit, drop_first=True, dtype=int,  prefix='categorical', prefix_sep='_')
y_credit = df_credit['default payment next month'].rename('y')
X_credit = df_credit.drop(columns=['ID', 'default payment next month'])

df_shoppers = pd.read_csv('data/online_shoppers_intention.csv')
categorical_cols_shoppers = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']
df_shoppers = pd.get_dummies(df_shoppers, columns=categorical_cols_shoppers, drop_first=True, dtype=int,  prefix='categorical', prefix_sep='_')
y_shoppers = df_shoppers['Revenue'].astype(int).rename('y')
X_shoppers = df_shoppers.drop(columns=['Revenue'])

X = [X_bank, X_credit, X_shoppers]
y = [y_bank, y_credit, y_shoppers]
dataset_names = ['bank', 'credit', 'shoppers']

In [ ]:
df_final = run_experiments(X, y, dataset_names)
df_final.to_csv('resultados_tcc_experimentos.csv', index=False)
df_final

# Comentários

- Não utilização das técnicas de pré processamento em variáveis dummy
- Remoção das técnicas de remoção